In [ ]:
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel

SEED = int(os.environ.get("REPRODUCIBLE_SEED", 42))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/datasets/anisoaraana/similarity"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_TRACK_B_PATH = DATA_DIR / "test_track_b.jsonl"
TEST_TRACK_B_LABELS = DATA_DIR / "test_track_b_labels.jsonl"

MODEL_NAME = os.environ.get("MODEL_NAME", "Qwen/Qwen3-Embedding-0.6B")
RUN_PREFIX = os.environ.get("RUN_PREFIX", "qwen3_06b_zero_shot")

# Qwen3-Embedding-0.6B supports long context
MAX_LEN = int(os.environ.get("MAX_LEN", 2048))
ENC_BS = int(os.environ.get("ENC_BS", 4))

# Qwen3 embeddings support Matryoshka-style dimensions up to 1024.
OUTPUT_DIM = int(os.environ.get("OUTPUT_DIM", 1024))

# For Track B, every story must get one role-independent vector.
# False for the cleanest zero-shot baseline, True for an instruction-aware variant.
USE_NARRATIVE_INSTRUCTION = os.environ.get("USE_NARRATIVE_INSTRUCTION", "0").strip().lower() in {"1", "true", "yes"}
NARRATIVE_INSTRUCTION = (
    "Instruct: Encode this story for narrative similarity. Focus on plot, actions, goals, outcomes, and theme.\n"
    "Story: "
)

OUT_NPY = OUT_DIR / f"{RUN_PREFIX}_track_b.npy"
DIAGNOSTICS_JSON = OUT_DIR / f"{RUN_PREFIX}_track_b_diagnostics.json"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"Track B path: {TEST_TRACK_B_PATH}")
print(f"MAX_LEN={MAX_LEN} ENC_BS={ENC_BS} OUTPUT_DIM={OUTPUT_DIM}")
print(f"Instruction-aware encoding: {USE_NARRATIVE_INSTRUCTION}")
print(f"Output: {OUT_NPY}")


## Load Track B Stories

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

rows_b = load_jsonl(TEST_TRACK_B_PATH)
texts_raw = [str(row["text"]).strip() for row in rows_b]

n_unique = len(set(texts_raw))
print(f"Loaded Track B stories: {len(texts_raw)}")
print(f"Unique texts: {n_unique}")
if n_unique != len(texts_raw):
    print("Warning: duplicate story texts detected. The .npy row order is still preserved.")

texts_to_encode = [NARRATIVE_INSTRUCTION + text for text in texts_raw] if USE_NARRATIVE_INSTRUCTION else texts_raw
print("Example text length:", len(texts_to_encode[0].split()))


## Load Qwen3-Embedding-0.6B

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
model.to(DEVICE)
model.eval()

print("Model loaded.")
print("Hidden size:", getattr(model.config, "hidden_size", None))


In [ ]:
def last_token_pool(last_hidden_states, attention_mask):
    # Works with both left and right padding. With left padding, the last token is always real.
    left_padding = bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item())
    if left_padding:
        return last_hidden_states[:, -1]

    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def encode_qwen3(texts, batch_size=4, max_len=2048, output_dim=1024):
    all_embeddings = []

    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc="Encoding Track B"):
            batch_texts = texts[start:start + batch_size]
            batch = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            ).to(DEVICE)

            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                outputs = model(**batch)
                emb = last_token_pool(outputs.last_hidden_state, batch["attention_mask"])

            if output_dim < emb.shape[1]:
                emb = emb[:, :output_dim]

            emb = F.normalize(emb.float(), p=2, dim=1)
            all_embeddings.append(emb.cpu().numpy().astype(np.float32))

            del batch, outputs, emb
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

    return np.vstack(all_embeddings).astype(np.float32)


X = encode_qwen3(texts_to_encode, batch_size=ENC_BS, max_len=MAX_LEN, output_dim=OUTPUT_DIM)
X = X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-12, None)

print("Embedding matrix:", X.shape)
print("Mean norm:", float(np.linalg.norm(X, axis=1).mean()))


## Save Track B Embeddings

In [ ]:
np.save(OUT_NPY, X.astype(np.float32))
print(f"Saved Track B embeddings: {OUT_NPY}")
print(f"Shape: {X.shape}")


In [ ]:
def norm_text(text):
    return " ".join(str(text).split())


def macro_f1_from_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int((y_pred & y_true).sum())
    tn = int((~y_pred & ~y_true).sum())
    fp = int((y_pred & ~y_true).sum())
    fn = int((~y_pred & y_true).sum())
    f1_pos = 2 * tp / (2 * tp + fp + fn + 1e-9)
    f1_neg = 2 * tn / (2 * tn + fn + fp + 1e-9)
    return (f1_pos + f1_neg) / 2.0, tp, tn, fp, fn


def evaluate_track_b(track_b_input_path, embeddings, labels_path):
    pred = pd.read_json(track_b_input_path, lines=True)
    if len(pred) != embeddings.shape[0]:
        raise ValueError(f"Track B rows={len(pred)}, embeddings rows={embeddings.shape[0]}")

    lookup = dict(zip(pred["text"].astype(str).map(norm_text), embeddings))
    gold = pd.read_json(labels_path, lines=True)

    y_true, y_pred = [], []
    missing = 0
    margins = []

    for _, row in gold.iterrows():
        anchor = lookup.get(norm_text(row["anchor_text"]))
        text_a = lookup.get(norm_text(row["text_a"]))
        text_b = lookup.get(norm_text(row["text_b"]))
        if anchor is None or text_a is None or text_b is None:
            missing += 1
            continue

        score_a = float(anchor.dot(text_a))
        score_b = float(anchor.dot(text_b))
        margins.append(score_a - score_b)
        y_pred.append(score_a > score_b)
        y_true.append(bool(row["text_a_is_closer"]))

    acc = accuracy_score(y_true, y_pred)
    macro_f1, tp, tn, fp, fn = macro_f1_from_binary(y_true, y_pred)
    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "n": int(len(y_true)),
        "missing": int(missing),
        "embedding_dim": int(embeddings.shape[1]),
        "mean_abs_margin": float(np.mean(np.abs(margins))) if margins else None,
    }


if TEST_TRACK_B_LABELS.exists():
    res_b = evaluate_track_b(TEST_TRACK_B_PATH, X, TEST_TRACK_B_LABELS)
    print("Track B label-file metrics:")
    print(json.dumps(res_b, indent=2))
else:
    res_b = None
    print("No Track B labels found; evaluation skipped.")


In [ ]:
diagnostics = {
    "model_name": MODEL_NAME,
    "run_prefix": RUN_PREFIX,
    "seed": SEED,
    "track_b_path": str(TEST_TRACK_B_PATH),
    "output_npy": str(OUT_NPY),
    "shape": list(X.shape),
    "max_len": MAX_LEN,
    "enc_bs": ENC_BS,
    "output_dim": OUTPUT_DIM,
    "use_narrative_instruction": USE_NARRATIVE_INSTRUCTION,
    "narrative_instruction": NARRATIVE_INSTRUCTION if USE_NARRATIVE_INSTRUCTION else None,
    "device": DEVICE,
    "dtype": str(DTYPE),
    "track_b_eval": res_b,
}

DIAGNOSTICS_JSON.write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")
print(f"Saved diagnostics: {DIAGNOSTICS_JSON}")
